# ZeroMQ: Publish-Subscribe Pattern
Wir haben einen zweiten (late) Subscriber.

Ggfs. Installation von ZeroMQ Package

In [1]:
!pip install pyzmq

Imports

In [2]:
import zmq
import threading
import time
from datetime import datetime

Publisher  
Änderungen zum einfachen Modell von vorher:  

`socket.setsockopt(zmq.LINGER, 0)`  
Standardmäßig wartet ZeroMQ beim Schließen, bis alle Nachrichten gesendet sind
In unserem Demo-Code kann das dazu führen, dass close() oder context.term() blockieren
Mit LINGER = 0 wird der Socket sofort geschlossen

`time.sleep(1)`  
PUB sendet sofort los
SUB braucht Zeit für:
connect()
SUBSCRIBE
Ohne Delay gehen die ersten Nachrichten verloren

`for i in range(15):`  
Mehr Nachrichten, damit der späte Subscriber sichtbar noch etwas bekommt
Sonst ist der Publisher schon fertig, bevor der late Subscriber richtig zuhören kann.

```
finally:
    socket.close()
    context.term()
```  
Garantiert sauberes Aufräumen, auch bei Fehlern (Threads und Jupyter sind so eine Sache...)

In [3]:
def run_publisher():
    context = zmq.Context()
    socket = context.socket(zmq.PUB)
    socket.setsockopt(zmq.LINGER, 0)
    socket.bind("tcp://*:12345")

    try:
        print("🟢 Publisher läuft...")
        time.sleep(1)  # Subscriber Zeit zum Verbinden geben

        for i in range(15):
            current_time = datetime.now().strftime("%H:%M:%S")
            message = f"TIME {current_time}"
            print("📤 Publisher sendet:", message)
            socket.send_string(message)
            time.sleep(1)

        print("🛑 Publisher beendet")
    finally:
        socket.close()
        context.term()

Subscriber
Änderungen:
`socket.setsockopt(zmq.RCVTIMEO, 3000)`  
recv() blockiert sonst für immer
Mit Timeout wirft ZeroMQ eine Exception (zmq.Again)

```
while True:
    message = socket.recv_string()
```  
Anzahl empfangener Nachrichten ist bei PUB/SUB nicht garantiert
feste Schleifen führen oft zu Deadlocks

```
except zmq.Again:
    print("⏱️ keine weiteren Nachrichten")
```  
sauberer Abbruch statt Hängen
Timeout wird hier behandelt

In [4]:
def run_subscriber(name="Subscriber"):
    context = zmq.Context()
    socket = context.socket(zmq.SUB)
    socket.setsockopt(zmq.LINGER, 0)
    socket.setsockopt(zmq.RCVTIMEO, 3000)
    socket.connect("tcp://127.0.0.1:12345")
    socket.setsockopt_string(zmq.SUBSCRIBE, "TIME")

    try:
        print(f"🔵 {name} wartet auf TIME-Nachrichten...")

        while True:
            message = socket.recv_string()
            print(f"📥 {name} empfängt:", message)

    except zmq.Again:
        print(f"⏱️ {name}: keine weiteren Nachrichten")
    finally:
        socket.close()
        context.term()
        print(f"🛑 {name} beendet")

Late Subscriber
Wir machen hier keine feste Anzahl von Nachrichten, sondern bis ggfs. Timeout auftritt.

`time.sleep(1.0)`  
Nach connect() + SUBSCRIBE dauert es kurz, bis:
Subscription beim Publisher ankommt
Ohne Delay:
erste Nachrichten gehen verloren
manchmal ALLE → scheinbar „hängt“
Klassisches Slow Joiner Problem

`socket.setsockopt(zmq.RCVTIMEO, 4000)`  
etwas länger als beim ersten Subscriber
LateSub hat weniger Zeitfenster
sonst endet er sofort wieder

In [5]:
def run_late_subscriber():
    context = zmq.Context()
    socket = context.socket(zmq.SUB)
    socket.setsockopt(zmq.LINGER, 0)
    socket.setsockopt(zmq.RCVTIMEO, 4000)
    socket.connect("tcp://127.0.0.1:12345")
    socket.setsockopt_string(zmq.SUBSCRIBE, "TIME")

    try:
        print("🟡 Später Subscriber gestartet")
        time.sleep(1.0)  # slow joiner etwas stärker abfedern

        while True:
            msg = socket.recv_string()
            print("📥 LateSub empfängt:", msg)

    except zmq.Again:
        print("⏱️ LateSub: keine weiteren Nachrichten")
    finally:
        socket.close()
        context.term()
        print("🛑 LateSub beendet")

Publisher und Subscriber starten.  
Wir müssen Threading verwenden.
`publisher_thread = threading.Thread(target=run_publisher)`  
kein daemon=True mehr
Daemon-Threads werden beim Programmende einfach gekillt
führen zu:
unvollständiger Ausgabe
komischem Verhalten im Notebook  

`publisher_thread.join()`  
Hauptprogramm wartet, bis Threads fertig sind
verhindert Zombie-Ausführung in Jupyter

In [ ]:
publisher_thread = threading.Thread(target=run_publisher)
subscriber_thread = threading.Thread(target=run_subscriber, args=("Sub1",))
late_sub_thread = threading.Thread(target=run_late_subscriber)

publisher_thread.start()
subscriber_thread.start()

time.sleep(4)
late_sub_thread.start()

publisher_thread.join()
subscriber_thread.join()
late_sub_thread.join()

print("✅ Alle Threads sauber beendet")

🟢 Publisher läuft...
🔵 Sub1 wartet auf TIME-Nachrichten...
📤 Publisher sendet: TIME 17:18:52
📥 Sub1 empfängt: TIME 17:18:52
📤 Publisher sendet: TIME 17:18:53
📥 Sub1 empfängt: TIME 17:18:53
📤 Publisher sendet: TIME 17:18:54
📥 Sub1 empfängt: TIME 17:18:54
📤 Publisher sendet:🟡 Später Subscriber gestartet
 TIME 17:18:55
📥 Sub1 empfängt: TIME 17:18:55
📤 Publisher sendet: TIME 17:18:56
📥 Sub1 empfängt: TIME 17:18:56
📥 LateSub empfängt: TIME 17:18:56
📤 Publisher sendet: TIME 17:18:57
📥 Sub1 empfängt: TIME 17:18:57
📥 LateSub empfängt: TIME 17:18:57
📤 Publisher sendet: TIME 17:18:58
📥 LateSub empfängt: TIME 17:18:58
📥 Sub1 empfängt: TIME 17:18:58
📤 Publisher sendet: TIME 17:18:59
📥 Sub1 empfängt: TIME 17:18:59
📥 LateSub empfängt: TIME 17:18:59
📤 Publisher sendet: TIME 17:19:00
📥 LateSub empfängt: TIME 17:19:00
📥 Sub1 empfängt: TIME 17:19:00
📤 Publisher sendet: TIME 17:19:01
📥 Sub1 empfängt: TIME 17:19:01
📥 LateSub empfängt: TIME 17:19:01
📤 Publisher sendet: TIME 17:19:02
📥 Sub1 empfängt: TIME 1

# Auswertung
Bei PUB/SUB gilt:  
- Publisher sendet an viele Empfänger
- Subscriber wählen Topics aus
- lose Kopplung
- asynchron
- Sender und Empfänger müssen nicht gleichzeitig starten
- aber: späte Subscriber verpassen alte Nachrichten: der späte Subscriber bekommt nur die späteren TIME-Nachrichten.